# PodaNauli
## Demonstrasi Analisis Service Gap Pariwisata Danau Toba

**Masalah:** ulasan wisata tersebar dan sulit diterjemahkan menjadi prioritas layanan yang dapat ditindaklanjuti.

**Solusi:** PodaNauli menggabungkan sentimen, complaint, aspek multi-label, metadata tempat, dan bukti ulasan menjadi Service Gap Ranking yang transparan.

**Pengguna:** pengelola destinasi, pemerintah daerah, pelaku UMKM, dan analis pariwisata.

> Hasil merupakan sistem pendukung analisis. Ranking bukan prediksi keuntungan, keputusan otomatis, atau pengganti validasi lapangan.

In [ ]:
import json
import sys
import warnings
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

PROJECT_ROOT = next(
    (
        candidate
        for candidate in (Path.cwd(), *Path.cwd().parents)
        if (candidate / "demo" / "demo_runtime.py").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Root proyek tidak ditemukan. Buka notebook dari folder repository atau folder demo.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from demo.demo_runtime import (
    display_data_quality_summary,
    display_metrics_summary,
    display_place_detail,
    display_prediction_result,
    display_service_gap_ranking,
    load_demo_bundle,
    predict_reviews,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 160)

DEMO_DIR = PROJECT_ROOT / "demo"
metrics = json.loads((DEMO_DIR / "demo_metrics.json").read_text(encoding="utf-8"))
quality = json.loads((DEMO_DIR / "demo_data_quality.json").read_text(encoding="utf-8"))
place_detail = json.loads((DEMO_DIR / "demo_place_detail.json").read_text(encoding="utf-8"))
demo_reviews = pd.read_csv(DEMO_DIR / "demo_reviews.csv")
service_gap = pd.read_csv(DEMO_DIR / "demo_service_gap.csv")
print("[OK] Paket demo lokal siap digunakan")

## 1. Karakter Dataset

Angka berikut dibaca dari artefak aktual, bukan ditulis ulang secara manual.

In [ ]:
display(display_data_quality_summary(quality))
display(pd.DataFrame(quality["raw_examples"]))

## 2. Transformasi Data

Teks kosong dan duplikat dipisahkan dari korpus NLP, variasi nama tempat disatukan, dan negasi seperti *tidak*, *kurang*, serta *belum* tetap dipertahankan.

In [ ]:
display(pd.DataFrame(quality["transformations"]))
display(pd.DataFrame([quality["cleaning_example"]]).rename(columns={"before": "Sebelum", "after": "Sesudah", "source": "Sumber"}))
display(pd.DataFrame([quality["entity_resolution_example"]]))
display(Image(filename=str(DEMO_DIR / "figures" / "data_transformation.png"), width=1050))

## 3. Cara Kerja Tiga Model

Sentimen menentukan polaritas umum, complaint menyaring indikasi keluhan, dan aspek menentukan bidang layanan yang dibahas. Satu ulasan dapat memiliki beberapa aspek.

In [ ]:
model_roles = pd.DataFrame([
    {"Model": "Sentimen", "Fungsi": "Polaritas umum", "Output": "Negatif / Netral / Positif", "Peran": "Memberi konteks umum"},
    {"Model": "Complaint", "Fungsi": "Deteksi keluhan", "Output": "Terdeteksi / Tidak / Tinjau", "Peran": "Menyaring bukti negatif"},
    {"Model": "Aspek", "Fungsi": "Klasifikasi multi-label", "Output": "Satu atau beberapa aspek", "Peran": "Menentukan bidang layanan"},
])
display(model_roles)
display(Image(filename=str(DEMO_DIR / "figures" / "pipeline_overview.png"), width=1050))

## 4. Muat Model Champion

Sel ini hanya memuat model tersimpan pada CPU. Tidak ada training atau akses internet.

In [ ]:
bundle = load_demo_bundle(print_status=True)

## 5. Inferensi Tiga Input Manual

Input A bersifat campuran, input B positif, dan input C memuat complaint akses.

In [ ]:
manual_inputs = [
    "Pemandangannya sangat indah, tetapi toilet kurang bersih dan area parkir sempit.",
    "Pelayanannya ramah, tempatnya nyaman, dan makanan disajikan dengan cepat.",
    "Akses jalan rusak, petunjuk arah kurang jelas, dan kendaraan sulit mencapai lokasi.",
]
manual_predictions = predict_reviews(manual_inputs, bundle)
display(display_prediction_result(manual_predictions))

## 6. Contoh Dataset Aktual

Contoh diambil dari locked test secara deterministik, telah dianonimkan, dan sengaja mencakup minimal satu kesalahan model.

In [ ]:
actual_view = demo_reviews[["place_name", "review_text", "expected_sentiment", "predicted_sentiment", "predicted_complaint", "predicted_aspects", "is_error_example"]].copy()
actual_view.columns = ["Tempat", "Ulasan", "Gold", "Prediksi", "Complaint", "Aspek", "Contoh error"]
display(actual_view)

## 7. Evaluasi Sentimen

Macro F1 digunakan karena distribusi kelas tidak seimbang. Recall negatif penting untuk mengurangi risiko keluhan yang terlewat.

In [ ]:
display(display_metrics_summary(metrics).query("Model == 'Sentimen'"))
display(Image(filename=str(DEMO_DIR / "figures" / "sentiment_confusion_matrix.png"), width=900))

## 8. Evaluasi Complaint

Sentimen mengukur polaritas umum, sedangkan complaint berfokus pada indikasi keluhan yang dapat ditindaklanjuti.

In [ ]:
display(display_metrics_summary(metrics).query("Model == 'Complaint'"))
display(pd.DataFrame([metrics["complaint"]["support"]]).rename(columns={"non_negative": "Non-complaint", "negative": "Complaint"}))
display(Image(filename=str(DEMO_DIR / "figures" / "complaint_confusion_matrix.png"), width=900))

## 9. Evaluasi Aspek Multi-label

Micro F1 menunjukkan performa agregat, Macro F1 pemerataan antaraspek, Hamming loss proporsi keputusan label yang salah, dan subset accuracy menuntut seluruh label tepat sekaligus.

In [ ]:
aspect_summary = pd.DataFrame([
    {"Metrik": "Micro F1", "Nilai": f"{metrics['aspect']['micro_f1']:.4f}"},
    {"Metrik": "Macro F1", "Nilai": f"{metrics['aspect']['macro_f1']:.4f}"},
    {"Metrik": "Hamming loss", "Nilai": f"{metrics['aspect']['hamming_loss']:.4f}"},
    {"Metrik": "Subset accuracy", "Nilai": f"{metrics['aspect']['subset_accuracy']:.4f}"},
])
display(aspect_summary)
display(Image(filename=str(DEMO_DIR / "figures" / "aspect_metrics.png"), width=1100))
print(f"Label 'lainnya': {metrics['aspect']['lainnya_total_gold']} gold total; support locked test {metrics['aspect']['lainnya_locked_test_support']}.")

## 10. Locked Test dan Pencegahan Leakage

Split dilakukan berdasarkan tempat. Tempat yang sama tidak tersebar bebas antar-split, sehingga model diuji pada kelompok tempat yang tidak digunakan saat training.

In [ ]:
split_rows = []
for model_name, model_key in [("Sentimen", "sentiment"), ("Aspek", "aspect")]:
    for split_name in ["train", "validation", "test"]:
        split_rows.append({"Model": model_name, "Split": split_name.title(), "Baris": metrics[model_key]["split_rows"][split_name], "Tempat": metrics[model_key]["split_places"][split_name]})
display(pd.DataFrame(split_rows))
print("Overlap tempat antarsplit:", metrics["sentiment"]["group_overlap_count"], "(sentimen),", metrics["aspect"]["group_overlap_count"], "(aspek)")

## 11. Service Gap Ranking

Ranking menunjukkan prioritas analisis berdasarkan bukti yang tersedia dalam dataset.

In [ ]:
display(display_service_gap_ranking(service_gap))
display(Image(filename=str(DEMO_DIR / "figures" / "service_gap_top10.png"), width=1100))

## 12. Detail Peringkat Teratas

Setiap ranking membawa jumlah bukti, reason code, confidence, dan potongan evidence yang telah dianonimkan.

In [ ]:
display(display_place_detail(place_detail))
display(pd.DataFrame({"Reason code": place_detail["reason_codes"], "Makna": place_detail["reason_labels"]}))
display(pd.DataFrame({"Potongan bukti": place_detail["evidence_snippets"]}))
print(place_detail["disclaimer"])

## 13. Validasi Manusia Top-20

Validasi terbatas pada 20 peringkat teratas dan satu validator. Validitas bukti tepat berada pada gate minimum 0,80.

In [ ]:
validation = metrics["service_gap_validation"]
display(pd.DataFrame([
    {"Metrik": "Evidence validity", "Nilai": f"{validation['evidence_validity']:.2f}"},
    {"Metrik": "Priority validity", "Nilai": f"{validation['priority_validity']:.2f}"},
    {"Metrik": "Overall validity", "Nilai": f"{validation['overall_validity']:.2f}"},
]))
display(Image(filename=str(DEMO_DIR / "figures" / "service_gap_validation.png"), width=1000))

## 14. Error Analysis

Kesalahan aktual tetap ditampilkan agar batas penggunaan model dapat dijelaskan secara jujur.

In [ ]:
display(pd.DataFrame(metrics["sentiment_error_examples"]))

## 15. Nilai Manfaat

In [ ]:
benefits = pd.DataFrame([
    {"Pengguna": "Pengelola", "Informasi": "Keluhan, aspek, dan bukti", "Keputusan yang didukung": "Prioritas perbaikan layanan"},
    {"Pengguna": "Pemerintah", "Informasi": "Pola lintas tempat dan lokasi", "Keputusan yang didukung": "Agenda verifikasi lapangan"},
    {"Pengguna": "UMKM", "Informasi": "Sinyal kebutuhan layanan", "Keputusan yang didukung": "Hipotesis layanan, bukan jaminan keuntungan"},
    {"Pengguna": "Analis", "Informasi": "Data terintegrasi dan metrik", "Keputusan yang didukung": "Analisis yang dapat ditelusuri"},
])
display(benefits)

## 16. Keterbatasan

In [ ]:
limitations = [
    "Human gold dibuat oleh satu anotator A01; inter-annotator agreement belum tersedia.",
    "Saran AI terlihat saat anotasi sehingga confirmation bias masih mungkin terjadi.",
    "Validasi ranking baru mencakup top-20 dan evidence validity tepat 0,80.",
    "Label aspek 'lainnya' hanya memiliki tiga gold secara keseluruhan.",
    "Metadata kosong bukan bukti bahwa fasilitas tidak tersedia.",
    "Hasil terbatas pada distribusi dataset ini dan tetap memerlukan validasi manusia.",
    "Sistem belum dinyatakan siap produksi.",
]
display(pd.DataFrame({"Keterbatasan yang perlu diperhatikan": limitations}))

## 17. Penutup

**PodaNauli mengubah suara wisatawan menjadi petunjuk berbasis data untuk membantu memahami prioritas layanan pariwisata Danau Toba.**

Model dan ranking pipeline memenuhi acceptance gate untuk analisis pada dataset ini, tetapi belum dinyatakan siap produksi.